# Test PVCNN

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import functools
from tqdm import tqdm
import random
import numpy as np
from typing import *

import torch
import torch.nn as nn
from torch import optim

from torch_pointcloud.models.pvcnn import PVConv, SharedMLP
from torch_pointcloud.models.pointnet2 import PointNetSA, PointNetFP, PointNetGlobalSA

## Utils

In [3]:
def _linear_bn_relu(in_channels, out_channels):
    return nn.Sequential(nn.Linear(in_channels, out_channels), nn.BatchNorm1d(out_channels), nn.ReLU(True))


def create_mlp_components(in_channels, out_channels, classifier=False, dim=2, width_multiplier=1):
    r = width_multiplier

    if dim == 1:
        block = _linear_bn_relu
    else:
        block = SharedMLP
    if not isinstance(out_channels, (list, tuple)):
        out_channels = [out_channels]
    if len(out_channels) == 0 or (len(out_channels) == 1 and out_channels[0] is None):
        return nn.Sequential(), in_channels, in_channels

    layers = []
    for oc in out_channels[:-1]:
        if oc < 1:
            layers.append(nn.Dropout(oc))
        else:
            oc = int(r * oc)
            layers.append(block(in_channels, oc))
            in_channels = oc
    if dim == 1:
        if classifier:
            layers.append(nn.Linear(in_channels, out_channels[-1]))
        else:
            layers.append(_linear_bn_relu(in_channels, int(r * out_channels[-1])))
    else:
        if classifier:
            layers.append(nn.Conv1d(in_channels, out_channels[-1], 1))
        else:
            layers.append(SharedMLP(in_channels, int(r * out_channels[-1])))
    return layers, out_channels[-1] if classifier else int(r * out_channels[-1])


def create_pointnet_components(blocks, in_channels, with_se=False, normalize=True, eps=0,
                               width_multiplier=1, voxel_resolution_multiplier=1):
    r, vr = width_multiplier, voxel_resolution_multiplier

    layers, concat_channels = [], 0
    for out_channels, num_blocks, voxel_resolution in blocks:
        out_channels = int(r * out_channels)
        if voxel_resolution is None:
            block = SharedMLP
        else:
            block = functools.partial(PVConv, kernel_size=3, resolution=int(vr * voxel_resolution),
                                      with_se=with_se, normalize=normalize, eps=eps)
        for _ in range(num_blocks):
            layers.append(block(in_channels, out_channels))
            in_channels = out_channels
            concat_channels += out_channels
    return layers, in_channels, concat_channels

## PVCNN

In [4]:
class PVCNN(nn.Module):
    blocks = ((64, 1, 32), (64, 2, 16), (128, 1, 16), (1024, 1, None))

    def __init__(self, num_classes, extra_feature_channels=6, width_multiplier=1, voxel_resolution_multiplier=1):
        super().__init__()
        self.in_channels = extra_feature_channels + 3

        layers, channels_point, concat_channels_point = create_pointnet_components(
            blocks=self.blocks, in_channels=self.in_channels, with_se=False,
            width_multiplier=width_multiplier, voxel_resolution_multiplier=voxel_resolution_multiplier
        )
        self.point_features = nn.ModuleList(layers)

        layers, channels_cloud = create_mlp_components(
            in_channels=channels_point, out_channels=[256, 128],
            classifier=False, dim=1, width_multiplier=width_multiplier)
        self.cloud_features = nn.Sequential(*layers)

        layers, _ = create_mlp_components(
            in_channels=(concat_channels_point + channels_cloud),
            out_channels=[512, 0.3, 256, 0.3, num_classes],
            classifier=True, dim=2, width_multiplier=width_multiplier
        )
        self.classifier = nn.Sequential(*layers)

    def forward(self, inputs):
        if isinstance(inputs, dict):
            inputs = inputs['features']

        coords = inputs[:, :3, :]
        out_features_list = []
        for i in range(len(self.point_features)):
            inputs, _ = self.point_features[i]((inputs, coords))
            out_features_list.append(inputs)
        # inputs: num_batches * 1024 * num_points -> num_batches * 1024 -> num_batches * 128
        inputs = self.cloud_features(inputs.max(dim=-1, keepdim=False).values)
        out_features_list.append(inputs.unsqueeze(-1).repeat([1, 1, coords.size(-1)]))
        return self.classifier(torch.cat(out_features_list, dim=1))

In [5]:
def train_one_epoch(model, optimizer, criterion, loader, device="cuda",
    log_interval: int = 5):
    model.train()
    
    loss_sum = 0
    
    pbar = tqdm(enumerate(loader), total=len(loader), desc="Training")
    for i, batch in pbar:
        optimizer.zero_grad()
        coords = batch["xyz_shifted"].to(device).transpose(1, 2)
        features = batch["rgb"].to(device).transpose(1, 2)
        target = batch["semantic"].to(device)
        B, _, N = coords.size()
        
        inputs = torch.cat([coords, features], dim=1)
        outputs = model(inputs)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
        
        loss_sum += loss
        
        if i % log_interval == 0:
            pbar.set_postfix({"train/loss_step": loss.item()})
    
    return {"train/loss_epoch": loss_sum / len(loader)}


In [6]:
device = "cuda"

model = PVCNN(num_classes=13, width_multiplier=0.125, extra_feature_channels=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [7]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [8]:
from torch_pointcloud.datasets import S3DIS
from torch.utils.data import DataLoader


train_dataset = S3DIS(root="../data")
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
)

In [9]:
for epoch in range(10):
    print(f"Epoch {epoch + 1}/{10}")
    metrics = {}
    # model, loader, criterion, optimizer, 
    train_metrics = train_one_epoch(model, optimizer, criterion, train_loader, device=device)
    metrics.update(train_metrics)
    
    print("Scores:", end=" ")
    print(" | ".join([f"{k}: {v:.4f}" for k, v in metrics.items()]))

Epoch 1/10


Training: 100%|██████████| 587/587 [00:15<00:00, 38.49it/s, train/loss_step=0.75] 

Scores: train/loss_epoch: 0.9792
Epoch 2/10



Training: 100%|██████████| 587/587 [00:14<00:00, 39.92it/s, train/loss_step=0.582]


Scores: train/loss_epoch: 0.6903
Epoch 3/10


...

Scores: train/loss_epoch: 0.4925
Epoch 9/10



Training: 100%|██████████| 587/587 [00:14<00:00, 39.64it/s, train/loss_step=0.427]

Scores: train/loss_epoch: 0.4798
Epoch 10/10



Training: 100%|██████████| 587/587 [00:14<00:00, 39.69it/s, train/loss_step=0.587]

Scores: train/loss_epoch: 0.4714
